# Get More Done in Less Time: 20 Essential Linux Commands for 80 % of Tasks

Run only in a disposable Linux VM with no host filesystem sharing. Install bash/coreutils/procps/iputils-ping/vim-tiny/OpenSSH and passwordless sudo inside that VM. The complete notebook creates isolated fixture paths; root operations affect only guest fixtures and package installation. It never kills unrelated processes or contacts an external SSH server. Original blocks remain verbatim in the nonexecuting appendix.

In [ ]:
import os, sys, platform, tempfile, subprocess, json, stat, shutil, socket, pwd, time
from pathlib import Path
if platform.system() != 'Linux':
    raise RuntimeError('Run this notebook in a disposable Linux VM, not on your host filesystem.')
guest_user = pwd.getpwuid(os.getuid()).pw_name
required = ['bash', 'ls', 'cp', 'mv', 'cat', 'chmod', 'chown', 'grep', 'top', 'ps', 'kill', 'sudo', 'du', 'tar', 'ping', 'vi', 'ssh', 'ssh-keygen']
assert all(shutil.which(name) for name in required)
workspace = tempfile.TemporaryDirectory(prefix='tdb-linux-commands-')
work = Path(workspace.name).resolve()
assert str(work).startswith('/tmp/tdb-linux-commands-')
results = {}
for section in range(1, 21):
    (work / str(section)).mkdir()
def fixture(section, name, text='example data\n'):
    path = work / str(section) / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text)
for name in ['visible.txt', '.hidden.txt']: fixture(1, name)
(work / '2' / 'Documents').mkdir()
fixture(5, 'myfile.txt'); fixture(5, 'my_folder/nested.txt')
fixture(6, 'file.txt'); fixture(6, 'my_folder/nested.txt')
fixture(7, 'file.txt'); (work / '7' / 'new_folder').mkdir()
for name, value in [('file.txt', 'Hello from a Linux fixture\n'), ('file1.txt', 'first\n'), ('file2.txt', 'second\n')]: fixture(8, name, value)
fixture(9, 'file.txt'); (work / '9' / 'dir').mkdir()
fixture(10, 'example.txt')
for name in ['mydirectory/data.txt', 'directory1/a.txt', 'directory2/b.txt']: fixture(16, name)
fixture(17, 'myfiles/a.txt', 'archive payload\n')
def run(script, section, key):
    cwd = (work / str(section)).resolve()
    assert cwd.is_relative_to(work)
    completed = subprocess.run(['bash', '-euo', 'pipefail', '-c', script], cwd=cwd, text=True, capture_output=True, timeout=90)
    results[key] = {'stdout': completed.stdout, 'stderr': completed.stderr, 'returncode': completed.returncode}
    if completed.returncode: raise RuntimeError(f'{key}: {completed.stderr}')
    print(completed.stdout, end='')
    if completed.stderr: print('Command stderr:', completed.stderr.strip())
    return completed
print('Python:', sys.version.split()[0])
print('Actual kernel:', platform.release(), platform.machine())
print('Disposable practice directory created; no host folders or user files are used.')

In [ ]:
script = 'ls .'
run(script, 1, 'example-0')
None

In [ ]:
script = 'ls -la .'
run(script, 1, 'example-1')
None

In [ ]:
script = 'cd Documents\npwd'
run(script, 2, 'example-2')
None

In [ ]:
script = 'cd /usr/bin\npwd'
run(script, 2, 'example-3')
None

In [ ]:
script = 'cd /usr/bin\ncd ..\npwd'
run(script, 2, 'example-4')
None

In [ ]:
script = 'pwd'
run(script, 3, 'example-5')
None

In [ ]:
script = 'mkdir my_folder\ntest -d my_folder\nprintf "Created my_folder\\n"'
run(script, 4, 'example-6')
None

In [ ]:
script = 'mkdir -p my_parent_folder/my_folder\ntest -d my_parent_folder/my_folder\nprintf "Created nested directories\\n"'
run(script, 4, 'example-7')
None

In [ ]:
script = 'rm -- myfile.txt\ntest ! -e myfile.txt\nprintf "Removed only the fixture file\\n"'
run(script, 5, 'example-8')
None

In [ ]:
script = 'rm -r -- my_folder\ntest ! -e my_folder\nprintf "Removed only the fixture directory\\n"'
run(script, 5, 'example-9')
None

In [ ]:
script = 'cp file.txt file_copy.txt\ncmp file.txt file_copy.txt\nprintf "File copy matches\\n"'
run(script, 6, 'example-10')
None

In [ ]:
script = 'cp -r my_folder my_folder_copy\ncmp my_folder/nested.txt my_folder_copy/nested.txt\nprintf "Directory copy matches\\n"'
run(script, 6, 'example-11')
None

In [ ]:
script = 'mv file.txt new_folder/\ntest -f new_folder/file.txt\nprintf "Moved fixture into existing directory\\n"'
run(script, 7, 'example-12')
None

In [ ]:
script = 'mv new_folder/file.txt new_file.txt\ntest -f new_file.txt\nprintf "Renamed the moved fixture\\n"'
run(script, 7, 'example-13')
None

In [ ]:
script = 'cat file.txt'
run(script, 8, 'example-14')
None

In [ ]:
script = 'cat file1.txt file2.txt > combined.txt\ncat combined.txt'
run(script, 8, 'example-15')
None

In [ ]:
script = 'chmod 644 file.txt\nstat -c "%a %n" file.txt'
run(script, 9, 'example-16')
None

In [ ]:
script = 'chmod 777 dir\nstat -c "%a %n" dir\nchmod 750 dir\nprintf "Restored the practice directory to mode 750\\n"'
run(script, 9, 'example-17')
None

In [ ]:
script = 'sudo -n chown nobody example.txt\nstat -c "%U:%G %n" example.txt'
run(script, 10, 'example-18')
None

In [ ]:
script = 'sudo -n chown nobody example.txt\nstat -c "%U:%G %n" example.txt'
run(script, 10, 'example-19')
None

In [ ]:
script = 'sudo -n chown nobody:nogroup example.txt\nstat -c "%U:%G %n" example.txt'
run(script, 10, 'example-20')
None

In [ ]:
script = "printf '%s\\n' apple banana orange grapefruit > example.txt\ncat example.txt"
run(script, 11, 'example-21')
None

In [ ]:
script = 'grep "an" example.txt'
run(script, 11, 'example-22')
None

In [ ]:
script = 'top -b -n 1 -p $$'
run(script, 12, 'top')
None

In [ ]:
script = 'ps aux'
run(script, 13, 'example-23')
None

In [ ]:
script = 'sleep 30 &\ntarget_pid=$!\ntrap \'kill "$target_pid" 2>/dev/null || true\' EXIT\nps aux | grep \'[s]leep\'\nps -p "$target_pid" -o comm='
run(script, 13, 'example-24')
None

In [ ]:
script = 'sleep 30 &\ntarget_pid=$!\nkill -TERM "$target_pid"\nstatus=0\nwait "$target_pid" || status=$?\ntest "$status" -eq 143\nprintf "Own child exited with SIGTERM status %s\\n" "$status"'
run(script, 14, 'example-25')
None

In [ ]:
script = 'sleep 30 &\ntarget_pid=$!\nkill -KILL "$target_pid"\nstatus=0\nwait "$target_pid" 2>/dev/null || status=$?\ntest "$status" -eq 137\nprintf "Own child exited with SIGKILL status %s\\n" "$status"'
run(script, 14, 'example-26')
None

In [ ]:
script = 'sudo -n apt-get install --no-install-recommends -y tree\ntree --version'
run(script, 15, 'example-27')
None

In [ ]:
script = 'du -sh .'
run(script, 16, 'example-28')
None

In [ ]:
script = 'du -h mydirectory'
run(script, 16, 'example-29')
None

In [ ]:
script = 'du -hc directory1 directory2'
run(script, 16, 'example-30')
None

In [ ]:
script = 'tar -cvf myfiles.tar myfiles/\ntar -tf myfiles.tar'
run(script, 17, 'example-31')
None

In [ ]:
script = 'tar -cvf myfiles.tar myfiles/'
run(script, 17, 'example-32')
None

In [ ]:
script = 'mkdir -p extracted\ntar -xvf myfiles.tar -C extracted\ncmp myfiles/a.txt extracted/myfiles/a.txt'
run(script, 17, 'example-33')
None

In [ ]:
script = 'tar -czvf myfiles.tar.gz myfiles/'
run(script, 17, 'example-34')
None

In [ ]:
script = 'mkdir -p extracted-gzip\ntar -xzvf myfiles.tar.gz -C extracted-gzip\ncmp myfiles/a.txt extracted-gzip/myfiles/a.txt'
run(script, 17, 'example-35')
None

In [ ]:
script = 'ping -c 3 127.0.0.1'
run(script, 18, 'example-36')
None

In [ ]:
script = 'ping -c 10 -i 0.2 127.0.0.1'
run(script, 18, 'example-37')
None

In [ ]:
script = "printf '%s\\n' 'i' 'Edited with real vi' '.' 'wq' | vi -e -s note.txt\ncat note.txt"
run(script, 19, 'vi')
None

In [ ]:
# A separate loopback-only SSH server with fixture keys and strict host checking.
ssh_fixture = tempfile.TemporaryDirectory(prefix='tdb-ssh-', dir=Path.home())
ssh_dir = Path(ssh_fixture.name)
client_key, host_key = ssh_dir / 'client', ssh_dir / 'host'
for key in [client_key, host_key]:
    subprocess.run(['ssh-keygen', '-q', '-t', 'ed25519', '-N', '', '-f', str(key)], check=True)
authorized = ssh_dir / 'authorized_keys'
authorized.write_text(client_key.with_suffix('.pub').read_text()); authorized.chmod(0o600)
with socket.socket() as probe:
    probe.bind(('127.0.0.1', 0)); ssh_port = probe.getsockname()[1]
configuration = ssh_dir / 'sshd_config'
configuration.write_text(f"Port {ssh_port}\nListenAddress 127.0.0.1\nHostKey {host_key}\nPidFile {ssh_dir / 'sshd.pid'}\nAuthorizedKeysFile {authorized}\nPasswordAuthentication no\nKbdInteractiveAuthentication no\nUsePAM yes\nPermitRootLogin no\nStrictModes yes\nAllowUsers {guest_user}\n")
known_hosts = ssh_dir / 'known_hosts'
known_hosts.write_text(f"[127.0.0.1]:{ssh_port} " + host_key.with_suffix('.pub').read_text())
subprocess.run(['sudo', '-n', 'mkdir', '-p', '/run/sshd'], check=True)
subprocess.run(['sudo', '-n', '/usr/sbin/sshd', '-f', str(configuration), '-E', str(ssh_dir / 'server.log')], check=True)
try:
    command = ['ssh', '-i', str(client_key), '-p', str(ssh_port), '-o', 'BatchMode=yes', '-o', 'IdentitiesOnly=yes', '-o', 'StrictHostKeyChecking=yes', '-o', f'UserKnownHostsFile={known_hosts}', f'{guest_user}@127.0.0.1', 'printf "Authenticated loopback SSH\n"; uname -s']
    ssh_result = subprocess.run(command, text=True, capture_output=True, timeout=20)
    if ssh_result.returncode:
        server_log = subprocess.check_output(['sudo', '-n', 'cat', str(ssh_dir / 'server.log')], text=True)
        raise RuntimeError(ssh_result.stderr + server_log)
    assert ssh_result.stdout == 'Authenticated loopback SSH\nLinux\n'
    print(ssh_result.stdout, end='')
finally:
    pid_file = ssh_dir / 'sshd.pid'
    if pid_file.exists():
        server_pid = int(pid_file.read_text())
        commandline = subprocess.check_output(['sudo', '-n', 'cat', f'/proc/{server_pid}/cmdline']).decode()
        assert str(configuration) in commandline
        subprocess.run(['sudo', '-n', 'kill', '--', str(server_pid)], check=True)
    ssh_fixture.cleanup()

In [ ]:
import pwd, grp
assert set(results) == {f'example-{i}' for i in range(38)} | {'top', 'vi'}
assert all(result['returncode'] == 0 for result in results.values())
assert results['example-22']['stdout'] == 'banana\norange\n'
assert (work / '8' / 'combined.txt').read_text() == 'first\nsecond\n'
assert (work / '6' / 'file.txt').read_bytes() == (work / '6' / 'file_copy.txt').read_bytes()
assert (work / '6' / 'my_folder/nested.txt').read_bytes() == (work / '6' / 'my_folder_copy/nested.txt').read_bytes()
assert not (work / '5' / 'myfile.txt').exists() and not (work / '5' / 'my_folder').exists()
assert (work / '7' / 'new_file.txt').exists() and not (work / '7' / 'file.txt').exists()
assert stat.S_IMODE((work / '9' / 'file.txt').stat().st_mode) == 0o644
assert stat.S_IMODE((work / '9' / 'dir').stat().st_mode) == 0o750
owned = (work / '10' / 'example.txt').stat()
assert owned.st_uid == pwd.getpwnam('nobody').pw_uid and owned.st_gid == grp.getgrnam('nogroup').gr_gid
for folder in ['extracted', 'extracted-gzip']:
    assert (work / '17' / folder / 'myfiles/a.txt').read_bytes() == (work / '17' / 'myfiles/a.txt').read_bytes()
assert (work / '19' / 'note.txt').read_text() == 'Edited with real vi\n'
assert '10 packets transmitted, 10 received' in results['example-37']['stdout']
assert ssh_result.stdout == 'Authenticated loopback SSH\nLinux\n'
# Restore guest fixture ownership before the temporary directory is removed.
subprocess.run(['sudo', '-n', 'chown', f'{os.getuid()}:{os.getgid()}', str(work / '10' / 'example.txt')], check=True)
print('PASS: all twenty Linux command families, actual permissions/ownership, copies, moves, deletion, grep, owned-child signals, archives, 10 loopback pings, vi save and strict-key loopback SSH.')
workspace.cleanup()

## Original source-code appendix

All original blocks, including syntax templates and historical output text, remain verbatim and nonexecuting:

```bash
ls ~
```

```bash
ls -l ~
```

```bash
cd Documents
```

```bash
cd /usr/bin
```

```bash
cd ..
```

```bash
pwd
```

```bash
mkdir my_folder
```

```bash
mkdir -p my_parent_folder/my_folder
```

```bash
rm myfile.txt
```

```bash
rm -r my_folder
```

```bash
cp file.txt file_copy.txt
```

```bash
cp -r my_folder my_folder_copy
```

```bash
mv file.txt new_folder/
```

```bash
mv file.txt new_file.txt
```

```bash
cat file.txt
```

```bash
cat file1.txt file2.txt > combined.txt
```

```bash
chmod 644 file.txt
```

```bash
chmod 777 dir
```

```bash
chown [new_owner][:[new_group]] file_name
```

```bash
sudo chown john example.txt
```

```bash
sudo chown john:admins example.txt
```

```bash
apple
banana
orange
grapefruit
```

```bash
banana
orange
grapefruit
```

```bash
ps aux
```

```bash
ps aux | grep firefox
```

```bash
kill 1234
```

```bash
kill -9 1234
```

```bash
sudo apt-get install <package_name>
```

```bash
du [options] [directory or file]
```

```bash
du -h /home/user/mydirectory
```

```bash
du -hc /home/user/directory1 /home/user/directory2
```

```bash
tar [options] [archive-filename] [file or directory to be archived]
```

```bash
tar -cvf myfiles.tar myfiles/
```

```bash
tar -xvf myfiles.tar
```

```bash
tar -czvf myfiles.tar.gz myfiles/
```

```bash
tar -xzvf myfiles.tar.gz
```

```bash
ping google.com
```

```bash
ping -c 10 -i 0.2 google.com
```

```bash
ssh username@remotehost
```